<a href="https://colab.research.google.com/github/mbiriri/Autonomous_Trailer/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autonomous Trailer Knowledge-Based Agent

Combines the 10-step knowledge engineering steps from Practical Exercise 3 with the 28-town road graph and search tree from `UCS_Trailer.ipynb` and the PEAS design in `design.md`.

## Step 1: Identify the Task
Build a knowledge-based agent for a self-driving freight trailer across Kenya's road network (focusing on **Nairobi to Kisumu**), using search tree pathfinding, distance measures, API data, and safety rules.

**PEAS**
- **Performance measure:** customer safety, fuel efficiency (least trip time), customer satisfaction, distance between towns
- **Environment:** roads, traffic laws, pedestrians, and customers
- **Actuators:** steering wheel, accelerator, brake, and horn
- **Sensors:** cameras and GPS

## Step 2: Assemble Relevant Knowledge
- 28 towns and 42 two-way roads (84 directed arcs) with highway names and distances from `UCS_Trailer.ipynb`
- Distance measures: **Euclidean**, **Manhattan**, and **Haversine**
- Rules for traffic signals, weighbridges, steep grades, and bridge clearance
- Live weather via **Open-Meteo** and hub lookup via **OpenStreetMap Nominatim**, with local fallbacks if offline

In [ ]:
import math
import heapq
import json
import urllib.request
import urllib.parse
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any

try:
    from bs4 import BeautifulSoup
    BS4_AVAILABLE = True
except ImportError:
    BS4_AVAILABLE = False

print(f"Ready: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | BeautifulSoup4: {BS4_AVAILABLE}")

Ready: 2026-11-19 18:13:03 | BeautifulSoup4: False


### External Data (APIs + Fallback)

In [ ]:
class ExternalKnowledgeHarvester:
    """Fetches weather and hub data via public APIs with a local fallback."""
    def __init__(self, timeout: int = 5):
        self.timeout = timeout
        self.headers = {"User-Agent": "AutonomousTrailerKBA-StudentProject/1.0"}

    def fetch_live_weather(self, lat: float, lon: float, location_name: str) -> Dict[str, Any]:
        url = (
            f"https://api.open-meteo.com/v1/forecast?"
            f"latitude={lat}&longitude={lon}&current=temperature_2m,precipitation,wind_speed_10m,weather_code"
        )
        try:
            req = urllib.request.Request(url, headers=self.headers)
            with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                data = json.loads(resp.read().decode("utf-8"))
                current = data.get("current", {})
                precip = current.get("precipitation", 0.0)
                wind = current.get("wind_speed_10m", 0.0)
                temp = current.get("temperature_2m", 22.0)

                condition = "Clear"
                if precip > 2.0:
                    condition = "Heavy_Rain"
                elif precip > 0.0:
                    condition = "Wet_Road"
                elif wind > 45.0:
                    condition = "High_Crosswind"

                return {
                    "location": location_name,
                    "source": "Open-Meteo Live API",
                    "temperature_c": temp,
                    "precipitation_mm": precip,
                    "wind_kmh": wind,
                    "road_weather_state": condition
                }
        except Exception:
            return {
                "location": location_name,
                "source": "Cached Telemetry Fallback",
                "temperature_c": 21.5,
                "precipitation_mm": 0.0,
                "wind_kmh": 14.2,
                "road_weather_state": "Clear"
            }

    def fetch_hub_metadata(self, query: str) -> Dict[str, Any]:
        encoded = urllib.parse.quote(query)
        url = f"https://nominatim.openstreetmap.org/search?q={encoded}&format=json&limit=1"
        try:
            req = urllib.request.Request(url, headers=self.headers)
            with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                results = json.loads(resp.read().decode("utf-8"))
                if results:
                    return {
                        "query": query,
                        "verified_name": results[0].get("display_name", query),
                        "lat": float(results[0]["lat"]),
                        "lon": float(results[0]["lon"]),
                        "source": "OSM Nominatim API"
                    }
        except Exception:
            pass
        return {"query": query, "verified_name": query, "source": "Local Corridor Registry"}

harvester = ExternalKnowledgeHarvester()
print(json.dumps(harvester.fetch_live_weather(-1.2921, 36.8219, "Nairobi"), indent=2))

{
  "location": "Nairobi",
  "source": "Cached Telemetry Fallback",
  "temperature_c": 21.5,
  "precipitation_mm": 0.0,
  "wind_kmh": 14.2,
  "road_weather_state": "Clear"
}


## Step 3: Define the Vocabulary
- **Towns (28):** `Nairobi`, `Kiambu`, `Limuru`, `Machakos`, `Thika`, `Murang'a`, `Karatina`, `Nyeri`, `Nanyuki`, `Embu`, `Nyahururu`, `Naivasha`, `Gilgil`, `Nakuru`, `Molo`, `Narok`, `Bomet`, `Kericho`, `Kapsabet`, `Eldoret`, `Kitale`, `Kisumu`, `Kisii`, `Homa Bay`, `Siaya`, `Kakamega`, `Bungoma`, `Busia`
- **Signals:** `Red`, `Yellow`, `Green`, `Flashing_Red`, `Flashing_Amber`, `None`
- **Weather:** `Clear`, `Wet_Road`, `Heavy_Rain`, `Fog`, `High_Crosswind`
- **Predicates:** `Light(color)`, `Road(u, v, km, name)`, `WeighbridgeAt(town)`, `WeatherAt(town, cond)`, `CanTraverse(trailer, u, v)`, `Action(percepts, cmd)`

## Step 4: Encode General Knowledge

In [ ]:
class DistanceMetrics:
    """Euclidean, Manhattan, and Haversine distance measures."""

    @staticmethod
    def euclidean(p1: Tuple[float, float], p2: Tuple[float, float]) -> float:
        return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)

    @staticmethod
    def manhattan(p1: Tuple[float, float], p2: Tuple[float, float]) -> float:
        return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

    @staticmethod
    def haversine(coord1: Tuple[float, float], coord2: Tuple[float, float]) -> float:
        R = 6371.0
        lat1, lon1 = map(math.radians, coord1)
        lat2, lon2 = map(math.radians, coord2)
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        return R * c


class SearchTreeNode:
    """Node in the search tree tracking state, parent, action/road, and costs."""
    def __init__(self, state: str, parent: Optional['SearchTreeNode'] = None,
                 road: Optional[str] = None, g_cost: float = 0.0, h_cost: float = 0.0):
        self.state = state
        self.parent = parent
        self.road = road
        self.g_cost = g_cost
        self.h_cost = h_cost
        self.f_cost = g_cost + h_cost
        self.depth = 0 if parent is None else parent.depth + 1

    def __lt__(self, other: 'SearchTreeNode') -> bool:
        return self.f_cost < other.f_cost

    def reconstruct_path(self) -> List[str]:
        path = []
        curr = self
        while curr is not None:
            path.append(curr.state)
            curr = curr.parent
        return list(reversed(path))

    def legs(self) -> List[Tuple[str, str, Optional[str], float, float]]:
        chain, node = [], self
        while node.parent is not None:
            chain.append(node)
            node = node.parent
        return [
            (n.parent.state, n.state, n.road,
             round(n.g_cost - n.parent.g_cost, 2), round(n.g_cost, 2))
            for n in reversed(chain)
        ]


class DirectedRoadGraph:
    """Directed graph of towns and road arcs from Practical Exercise 3 (UCS_Trailer.ipynb)."""
    def __init__(self):
        self.nodes: Dict[str, Dict[str, Any]] = {}
        self.edges: Dict[str, List[Dict[str, Any]]] = {}

    def add_town(self, name: str, lat: float, lon: float, grid_xy: Tuple[float, float],
                 weighbridge: bool = False):
        self.nodes[name] = {
            "lat": lat,
            "lon": lon,
            "grid_xy": grid_xy,
            "weighbridge": weighbridge,
            "weather": "Clear"
        }
        self.edges.setdefault(name, [])

    def add_road(self, a: str, b: str, distance_km: float, road_name: str,
                 bidirectional: bool = True, speed_limit: int = 80,
                 max_tons: float = 50.0, clearance_m: float = 5.2,
                 steep_grade: bool = False, open_road: bool = True):
        if a not in self.nodes or b not in self.nodes:
            raise KeyError(f"Unknown town in road: {a} - {b}")
        self.edges[a].append({
            "to": b,
            "distance_km": distance_km,
            "road": road_name,
            "speed_limit": speed_limit,
            "max_tons": max_tons,
            "clearance_m": clearance_m,
            "steep_grade": steep_grade,
            "open_road": open_road
        })
        if bidirectional:
            self.edges[b].append({
                "to": a,
                "distance_km": distance_km,
                "road": road_name,
                "speed_limit": speed_limit,
                "max_tons": max_tons,
                "clearance_m": clearance_m,
                "steep_grade": steep_grade,
                "open_road": open_road
            })

    def get_heuristic(self, u: str, goal: str, metric: str = "haversine") -> float:
        n1, n2 = self.nodes[u], self.nodes[goal]
        if metric == "haversine":
            return DistanceMetrics.haversine((n1["lat"], n1["lon"]), (n2["lat"], n2["lon"]))
        elif metric == "euclidean":
            return DistanceMetrics.euclidean(n1["grid_xy"], n2["grid_xy"])
        elif metric == "manhattan":
            return DistanceMetrics.manhattan(n1["grid_xy"], n2["grid_xy"])
        elif metric == "none":
            return 0.0  # Reduces to Uniform Cost Search (UCS)
        raise ValueError(f"Unknown metric: {metric}")


TRAFFIC_SIGNAL_RULES = {
    "Red": "FULL_STOP — Engage air brakes and hold behind stop line.",
    "Yellow": "DECELERATE_TO_STOP — Apply engine retarder; stop unless within heavy-trailer braking distance.",
    "Green": "PROCEED — Scan cross-traffic, check trailer swing clearance, and maintain target speed.",
    "Flashing_Red": "STOP_AND_YIELD — Come to a complete stop, check intersection is clear, then proceed.",
    "Flashing_Amber": "PROCEED_WITH_CAUTION — Slow down and watch merging/pedestrian traffic.",
    "None": "UNCONTROLLED_INTERSECTION — Treat as yield; slow to 25 km/h and scan sensors."
}

WEATHER_SPEED_MULTIPLIERS = {
    "Clear": 1.0,
    "Wet_Road": 0.75,
    "Heavy_Rain": 0.55,
    "Fog": 0.50,
    "High_Crosswind": 0.65
}

print("Distance measures, search tree node, directed graph, and rules ready.")

Distance measures, search tree node, directed graph, and rules ready.


## Step 5: Add Facts & Build the KB

In [ ]:
class TrailerKnowledgeBase:
    """Knowledge base storing rules, trailer specs, and the 28-town directed graph."""
    def __init__(self):
        self.signal_rules: Dict[str, str] = dict(TRAFFIC_SIGNAL_RULES)
        self.weather_multipliers: Dict[str, float] = dict(WEATHER_SPEED_MULTIPLIERS)
        self.custom_rules: Dict[str, str] = {}
        self.graph = DirectedRoadGraph()
        self.trailer_profile: Dict[str, Any] = {
            "id": "TRL-KE-001",
            "gross_weight_tons": 38.0,
            "height_m": 4.3,
            "cargo_type": "Standard_Load"
        }
        self.current_percepts: Dict[str, Any] = {}

    def add_new_fact(self, category: str, key: str, value: Any):
        cat = category.lower()
        if cat == "signal":
            self.signal_rules[key] = value
        elif cat == "weather":
            self.weather_multipliers[key] = float(value)
        elif cat == "rule":
            self.custom_rules[key] = value
        elif cat == "percept":
            self.current_percepts[key] = value
        elif cat == "trailer":
            self.trailer_profile[key] = value
        else:
            self.custom_rules[f"{category}:{key}"] = str(value)
        print(f"[KB TELL] [{category}] {key} -> {value}")

    def set_road_status(self, a: str, b: str, open_road: bool, reason: str = ""):
        updated = False
        for u, v in [(a, b), (b, a)]:
            for edge in self.graph.edges.get(u, []):
                if edge["to"] == v:
                    edge["open_road"] = open_road
                    updated = True
        if updated:
            state = "OPEN" if open_road else f"CLOSED ({reason})"
            print(f"[KB GRAPH] {a} <-> {b}: {state}")
        else:
            print(f"[KB WARNING] Road {a} - {b} not found.")

    def show_kb(self):
        print("\n--- Knowledge Base Summary ---")
        print("Trailer:", self.trailer_profile)
        print(f"Towns: {len(self.graph.nodes)} | Directed arcs: {sum(len(v) for v in self.graph.edges.values())}")
        print("Rules:", self.custom_rules)
        print("------------------------------\n")


kb = TrailerKnowledgeBase()

# 28 towns from Practical Exercise 3 (UCS_Trailer.ipynb) with approximate (lat, lon) and grid coords
towns_data = [
    ("Nairobi",    -1.2921, 36.8219, (440.0, 300.0), True),
    ("Kiambu",     -1.1714, 36.8356, (442.0, 314.0), False),
    ("Limuru",     -1.1136, 36.6425, (418.0, 320.0), False),
    ("Machakos",   -1.5177, 37.2634, (490.0, 275.0), False),
    ("Thika",      -1.0332, 37.0693, (468.0, 328.0), False),
    ("Murang'a",   -0.7839, 37.0458, (465.0, 356.0), False),
    ("Karatina",   -0.4867, 37.1292, (475.0, 390.0), False),
    ("Nyeri",      -0.4201, 36.9476, (454.0, 397.0), False),
    ("Nanyuki",     0.0167, 37.0731, (468.0, 445.0), False),
    ("Embu",       -0.5318, 37.4506, (510.0, 385.0), False),
    ("Nyahururu",   0.0424, 36.3673, (390.0, 448.0), False),
    ("Naivasha",   -0.7172, 36.4310, (397.0, 364.0), False),
    ("Gilgil",     -0.4985, 36.3250, (385.0, 388.0), False),
    ("Nakuru",     -0.3031, 36.0800, (358.0, 410.0), True),
    ("Molo",       -0.2478, 35.7347, (320.0, 416.0), False),
    ("Narok",      -1.0875, 35.8686, (335.0, 323.0), False),
    ("Bomet",      -0.7813, 35.3416, (276.0, 357.0), False),
    ("Kericho",    -0.3676, 35.2831, (270.0, 403.0), False),
    ("Kapsabet",    0.2012, 35.1050, (250.0, 466.0), False),
    ("Eldoret",     0.5143, 35.2698, (268.0, 500.0), True),
    ("Kitale",      1.0157, 35.0062, (239.0, 556.0), False),
    ("Kisumu",     -0.0917, 34.7680, (212.0, 433.0), True),
    ("Kisii",      -0.6773, 34.7796, (214.0, 368.0), False),
    ("Homa Bay",   -0.5273, 34.4571, (178.0, 385.0), False),
    ("Siaya",       0.0607, 34.2882, (159.0, 450.0), False),
    ("Kakamega",    0.2827, 34.7519, (211.0, 475.0), False),
    ("Bungoma",     0.5635, 34.5606, (190.0, 506.0), False),
    ("Busia",       0.4608, 34.1117, (140.0, 495.0), False),
]

for name, lat, lon, gxy, wb in towns_data:
    kb.graph.add_town(name, lat, lon, gxy, weighbridge=wb)

# Roads from UCS_Trailer.ipynb: (from, to, distance_km, road)
roads_data = [
    ("Nairobi", "Kiambu", 14, "A2/Kiambu Rd"),
    ("Nairobi", "Limuru", 30, "A104"),
    ("Nairobi", "Thika", 42, "A2/Thika Superhighway"),
    ("Nairobi", "Machakos", 64, "A109 / Machakos Rd"),
    ("Kiambu", "Limuru", 20, "Kiambu-Limuru Rd"),
    ("Kiambu", "Thika", 35, "Kiambu-Ruiru-Thika"),
    ("Thika", "Machakos", 50, "Thika-Kangundo-Machakos"),
    ("Thika", "Murang'a", 40, "B-road via Makuyu"),
    ("Thika", "Nyeri", 100, "A2"),
    ("Thika", "Embu", 100, "Thika-Kenol-Embu"),
    ("Murang'a", "Karatina", 45, "B-road"),
    ("Karatina", "Nyeri", 25, "A2"),
    ("Karatina", "Embu", 55, "B-road via Kerugoya"),
    ("Nyeri", "Nanyuki", 60, "A2"),
    ("Nyeri", "Nyahururu", 80, "B-road"),
    ("Limuru", "Naivasha", 60, "A104"),
    ("Naivasha", "Gilgil", 30, "A104"),
    ("Gilgil", "Nakuru", 35, "A104"),
    ("Nakuru", "Nyahururu", 75, "B-road"),
    ("Nakuru", "Molo", 45, "A104"),
    ("Nairobi", "Narok", 140, "A104/B3 via Suswa"),
    ("Naivasha", "Narok", 105, "Naivasha-Mai Mahiu-Narok"),
    ("Narok", "Bomet", 85, "B-road"),
    ("Bomet", "Kericho", 45, "B-road"),
    ("Bomet", "Kisii", 85, "B-road"),
    ("Molo", "Kericho", 70, "A104/B1 via Londiani"),
    ("Molo", "Eldoret", 110, "A104 via Timboroa"),
    ("Kericho", "Kisumu", 80, "A1 via Muhoroni"),
    ("Eldoret", "Kapsabet", 40, "B-road"),
    ("Kapsabet", "Kisumu", 75, "B-road"),
    ("Eldoret", "Kitale", 70, "A104/B-road"),
    ("Eldoret", "Kakamega", 105, "A1/B-road"),
    ("Kisumu", "Kisii", 110, "A1/B-road"),
    ("Kisumu", "Homa Bay", 100, "B-road"),
    ("Homa Bay", "Kisii", 65, "B-road"),
    ("Kisumu", "Siaya", 60, "A1/B-road"),
    ("Siaya", "Busia", 50, "B-road"),
    ("Kisumu", "Kakamega", 50, "A1"),
    ("Kakamega", "Bungoma", 50, "A1 via Webuye"),
    ("Bungoma", "Busia", 50, "A1/B-road"),
    ("Bungoma", "Kitale", 60, "B-road"),
    ("Kisumu", "Busia", 105, "A1 via Kakamega-Mumias"),
]

for a, b, km, road in roads_data:
    steep = (a, b) in [("Limuru", "Naivasha"), ("Naivasha", "Narok"), ("Molo", "Eldoret")]
    clearance = 4.6 if road.startswith("B-road") else 5.2
    kb.graph.add_road(a, b, km, road, bidirectional=True, steep_grade=steep, clearance_m=clearance)

kb.add_new_fact("rule", "Weighbridge_Check", "Trailers > 3.5t stop at weighbridges at 15 km/h.")
kb.add_new_fact("rule", "Steep_Grade_Speed", "Cap speed at 45 km/h and use engine braking on steep grades.")
kb.add_new_fact("rule", "Clearance_Buffer", "Keep at least 0.25m clearance below bridges.")

for hub in ["Nairobi", "Naivasha", "Nakuru", "Kisumu"]:
    node = kb.graph.nodes[hub]
    w = harvester.fetch_live_weather(node["lat"], node["lon"], hub)
    node["weather"] = w["road_weather_state"]

kb.show_kb()

[KB TELL] [rule] Weighbridge_Check -> Trailers > 3.5t stop at weighbridges at 15 km/h.
[KB TELL] [rule] Steep_Grade_Speed -> Cap speed at 45 km/h and use engine braking on steep grades.
[KB TELL] [rule] Clearance_Buffer -> Keep at least 0.25m clearance below bridges.

--- Knowledge Base Summary ---
Trailer: {'id': 'TRL-KE-001', 'gross_weight_tons': 38.0, 'height_m': 4.3, 'cargo_type': 'Standard_Load'}
Towns: 28 | Directed arcs: 84
Rules: {'Weighbridge_Check': 'Trailers > 3.5t stop at weighbridges at 15 km/h.', 'Steep_Grade_Speed': 'Cap speed at 45 km/h and use engine braking on steep grades.', 'Clearance_Buffer': 'Keep at least 0.25m clearance below bridges.'}
------------------------------



## Steps 6 & 7: Inference, Search Tree & Queries

In [ ]:
class AutonomousTrailerAgent:
    """Knowledge-based trailer agent using search tree planning and safety rules."""
    def __init__(self, knowledge_base: TrailerKnowledgeBase):
        self.kb = knowledge_base

    def can_traverse(self, edge: Dict[str, Any]) -> Tuple[bool, str]:
        if not edge["open_road"]:
            return False, "road closed"
        if self.kb.trailer_profile["gross_weight_tons"] > edge["max_tons"]:
            return False, "weight limit exceeded"
        if self.kb.trailer_profile["height_m"] + 0.25 > edge["clearance_m"]:
            return False, "clearance too low"
        return True, "ok"

    def safe_speed(self, town: str, edge: Dict[str, Any]) -> float:
        weather = self.kb.graph.nodes[town].get("weather", "Clear")
        mult = self.kb.weather_multipliers.get(weather, 0.7)
        spd = edge["speed_limit"] * mult
        if edge["steep_grade"]:
            spd = min(spd, 45.0)
        return round(spd, 1)

    def search_route(self, start: str, goal: str, metric: str = "haversine",
                     optimize_by: str = "distance") -> Dict[str, Any]:
        """
        Search tree over the directed graph.
        Use metric='none' for Uniform Cost Search (UCS), or
        metric='haversine' / 'euclidean' / 'manhattan' for A* search.
        """
        if start not in self.kb.graph.nodes or goal not in self.kb.graph.nodes:
            return {"success": False, "reason": "Unknown start or goal town."}

        h0 = self.kb.graph.get_heuristic(start, goal, metric)
        if optimize_by == "time":
            h0 /= 80.0

        root = SearchTreeNode(start, g_cost=0.0, h_cost=h0)
        frontier: List[SearchTreeNode] = [root]
        best_g: Dict[str, float] = {start: 0.0}
        explored: List[str] = []
        pruned: List[str] = []

        while frontier:
            current = heapq.heappop(frontier)
            if current.state == goal:
                return {
                    "success": True,
                    "path": current.reconstruct_path(),
                    "legs": current.legs(),
                    "cost": round(current.g_cost, 2),
                    "expanded_count": len(explored),
                    "explored": explored,
                    "pruned": pruned,
                    "goal_node": current
                }

            if current.state in explored:
                continue
            explored.append(current.state)

            for edge in self.kb.graph.edges.get(current.state, []):
                nxt = edge["to"]
                ok, reason = self.can_traverse(edge)
                if not ok:
                    pruned.append(f"{current.state} -> {nxt} ({reason})")
                    continue

                spd = self.safe_speed(current.state, edge)
                step_cost = edge["distance_km"] if optimize_by == "distance" else (edge["distance_km"] / spd)
                tentative_g = current.g_cost + step_cost

                if nxt not in best_g or tentative_g < best_g[nxt]:
                    best_g[nxt] = tentative_g
                    h_val = self.kb.graph.get_heuristic(nxt, goal, metric)
                    if optimize_by == "time":
                        h_val /= 80.0
                    child = SearchTreeNode(nxt, parent=current, road=edge["road"],
                                           g_cost=tentative_g, h_cost=h_val)
                    heapq.heappush(frontier, child)

        return {"success": False, "reason": "No valid path found.", "pruned": pruned}

    def decide_action(self, percepts: Dict[str, Any]) -> Dict[str, Any]:
        """Maps current sensor percepts (cameras, GPS, speed, obstacle, weather) to actions."""
        self.kb.current_percepts = percepts
        light = percepts.get("traffic_light", "Green")
        for k in self.kb.signal_rules:
            if k.lower() == light.lower():
                light = k
                break

        town = percepts.get("town", "Nairobi")
        speed_kmh = percepts.get("speed_kmh", 60.0)
        obstacle_m = percepts.get("obstacle_distance_m", 999.0)
        steep_grade = percepts.get("steep_grade", False)
        weather = percepts.get("weather", self.kb.graph.nodes.get(town, {}).get("weather", "Clear"))

        mu = 0.7 if weather == "Clear" else 0.4
        v_ms = speed_kmh / 3.6
        stopping_m = round((v_ms ** 2) / (2 * mu * 9.81) + v_ms * 1.2, 1)

        actions = []
        if obstacle_m <= stopping_m * 1.2:
            actions.append(f"BRAKE: Obstacle at {obstacle_m}m (stopping dist: {stopping_m}m).")
        actions.append(f"SIGNAL [{light}]: {self.kb.signal_rules.get(light, 'STOP — Unknown signal, halt safely.')}")

        if town in self.kb.graph.nodes and self.kb.graph.nodes[town]["weighbridge"]:
            actions.append(f"WEIGHBRIDGE [{town}]: Enter inspection lane at 15 km/h.")

        target_spd = round(80.0 * self.kb.weather_multipliers.get(weather, 0.7), 1)
        if steep_grade:
            target_spd = min(target_spd, 45.0)
            actions.append("GRADE: Steep section — use engine braking, max 45 km/h.")
        actions.append(f"SPEED: Target {target_spd} km/h ({weather}, stopping ~{stopping_m}m).")

        return {"town": town, "stopping_distance_m": stopping_m, "actions": actions}

agent = AutonomousTrailerAgent(kb)
print("Agent ready.")

Agent ready.


## Step 8: Querying the KB

In [ ]:
# 1. Nairobi -> Kisumu using UCS (metric='none') vs A* with Haversine, Euclidean, and Manhattan
for m in ["none", "haversine", "euclidean", "manhattan"]:
    res = agent.search_route("Nairobi", "Kisumu", metric=m, optimize_by="distance")
    print(f"Metric={m:9} | Cost={res['cost']} km | Expanded={res['expanded_count']:2} | Path={' -> '.join(res['path'])}")

print("\nLegs for Nairobi -> Kisumu:")
ucs_res = agent.search_route("Nairobi", "Kisumu", metric="none", optimize_by="distance")
for a, b, road, leg, total in ucs_res["legs"]:
    print(f"  {a:10} -> {b:10} | {road:28} | {leg:5} km | {total:5} km")

# 2. Tactical percept query
print("\nTactical percept check:")
tact = agent.decide_action({
    "town": "Nakuru",
    "traffic_light": "Yellow",
    "speed_kmh": 65.0,
    "obstacle_distance_m": 150.0,
    "steep_grade": False
})
for act in tact["actions"]:
    print(" ", act)

# 3. Over-height trailer constraint check (4.60m -> 4.85m needed > 4.6m B-road clearance)
print("\nClearance constraint check:")
kb.add_new_fact("trailer", "height_m", 4.60)
blocked = agent.search_route("Nairobi", "Kisumu", metric="haversine")
print("  Path with 4.60m trailer:", " -> ".join(blocked["path"]), "| Cost:", blocked["cost"], "km")
print("  Pruned B-road sample:", blocked["pruned"][:3])
kb.add_new_fact("trailer", "height_m", 4.30)

Metric=none      | Cost=350.0 km | Expanded=21 | Path=Nairobi -> Limuru -> Naivasha -> Gilgil -> Nakuru -> Molo -> Kericho -> Kisumu
Metric=haversine | Cost=350.0 km | Expanded=12 | Path=Nairobi -> Limuru -> Naivasha -> Gilgil -> Nakuru -> Molo -> Kericho -> Kisumu
Metric=euclidean | Cost=350.0 km | Expanded=12 | Path=Nairobi -> Limuru -> Naivasha -> Gilgil -> Nakuru -> Molo -> Kericho -> Kisumu
Metric=manhattan | Cost=350.0 km | Expanded= 7 | Path=Nairobi -> Limuru -> Naivasha -> Gilgil -> Nakuru -> Molo -> Kericho -> Kisumu

Legs for Nairobi -> Kisumu:
  Nairobi    -> Limuru     | A104                         |  30.0 km |  30.0 km
  Limuru     -> Naivasha   | A104                         |  60.0 km |  90.0 km
  Naivasha   -> Gilgil     | A104                         |  30.0 km | 120.0 km
  Gilgil     -> Nakuru     | A104                         |  35.0 km | 155.0 km
  Nakuru     -> Molo       | A104                         |  45.0 km | 200.0 km
  Molo       -> Kericho    | A104/B1 vi

### Interactive Mode
Set `auto_demo=False` for interactive prompts.

In [ ]:
def interactive_console(auto_demo: bool = True):
    if auto_demo:
        for start, goal, sig in [("Nairobi", "Eldoret", "Green"), ("Limuru", "Naivasha", "Red")]:
            r = agent.search_route(start, goal, metric="haversine", optimize_by="time")
            a = agent.decide_action({"town": start, "traffic_light": sig, "steep_grade": (start == "Limuru")})
            print(f"{start} -> {goal} ({r['cost']} hrs): {' -> '.join(r['path'])}")
            print("  Action:", a["actions"][0])
    else:
        print("Towns:", list(kb.graph.nodes.keys()))
        s = input("Start town: ").strip()
        g = input("Goal town: ").strip()
        sig = input("Signal: ").strip()
        print(agent.search_route(s, g))
        print(agent.decide_action({"town": s, "traffic_light": sig}))

interactive_console(auto_demo=True)

Nairobi -> Eldoret (5.53 hrs): Nairobi -> Limuru -> Naivasha -> Gilgil -> Nakuru -> Molo -> Eldoret
  Action: SIGNAL [Green]: PROCEED — Scan cross-traffic, check trailer swing clearance, and maintain target speed.
Limuru -> Naivasha (1.33 hrs): Limuru -> Naivasha
  Action: SIGNAL [Red]: FULL_STOP — Engage air brakes and hold behind stop line.


## Step 9: Debugging the KB

In [ ]:
for sig in ["Red", "Yellow", "Green", "Blue", "Teal", "None"]:
    out = agent.decide_action({"town": "Naivasha", "traffic_light": sig, "speed_kmh": 50.0})
    print(f"Signal {sig:8} -> {out['actions'][0]}")

print("\nRoad closure check (Kericho - Kisumu):")
kb.set_road_status("Kericho", "Kisumu", False, "Landslide")
alt = agent.search_route("Nairobi", "Kisumu", metric="none")
print("  Rerouted path:", " -> ".join(alt["path"]), "| Cost:", alt["cost"], "km")
kb.set_road_status("Kericho", "Kisumu", True)

Signal Red      -> SIGNAL [Red]: FULL_STOP — Engage air brakes and hold behind stop line.
Signal Yellow   -> SIGNAL [Yellow]: DECELERATE_TO_STOP — Apply engine retarder; stop unless within heavy-trailer braking distance.
Signal Green    -> SIGNAL [Green]: PROCEED — Scan cross-traffic, check trailer swing clearance, and maintain target speed.
Signal Blue     -> SIGNAL [Blue]: STOP — Unknown signal, halt safely.
Signal Teal     -> SIGNAL [Teal]: STOP — Unknown signal, halt safely.
Signal None     -> SIGNAL [None]: UNCONTROLLED_INTERSECTION — Treat as yield; slow to 25 km/h and scan sensors.

Road closure check (Kericho - Kisumu):
[KB GRAPH] Kericho <-> Kisumu: CLOSED (Landslide)
  Rerouted path: Nairobi -> Narok -> Bomet -> Kisii -> Kisumu | Cost: 420.0 km
[KB GRAPH] Kericho <-> Kisumu: OPEN


## Step 10: Reflection
- **Challenges:** Handling both shortest-distance search (matching the 350 km Nairobi–Kisumu UCS route) and heavy-trailer safety rules like braking distance, bridge clearance, and weather speed limits. External APIs also need fallbacks when offline.
- **Improvements:** Add sensor noise/uncertainty, live traffic/signal data, and multi-trailer coordination.